# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes


In [10]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)

DATA_PATH = "/content/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)

df.head()

Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

In [11]:
leak_cols = [
    "trend_direction",
    "trend_pct",
    "is_declining_label"
]

features = df.drop(columns=leak_cols, errors="ignore")

features.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,NaN,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,NaN,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,NaN,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,NaN,NaN,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,NaN,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,NaN,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5


In [12]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[
        -1,
        30,
        90,
        180,
        365,
        np.inf
    ],
    labels=[
        "0-30 days",
        "31-90 days",
        "91-180 days",
        "181-365 days",
        "365+ days"
    ]
)

staleness_check = (
    df.groupby("staleness_bucket", observed=True)
    .agg(
        n=("content_id","count"),
        avg_trend=("trend_pct","mean")
    )
    .reset_index()
)

staleness_check

,staleness_bucket,n,avg_trend
0,0-30 days,20480,0.784405
1,31-90 days,175,-7.373054
2,91-180 days,9171,-15.683224
3,181-365 days,169,-4.718462
4,365+ days,5,-96.166667


Verdict: CONFIRMED

Older content shows a stronger negative trend pattern.
This supports using staleness as a refresh priority signal.

In [13]:
df["volume_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=5,
    duplicates="drop"
)


volume_check = (
    df.groupby("volume_bucket", observed=True)
    .agg(
        n=("content_id","count"),
        avg_trend=("trend_pct","mean")
    )
    .reset_index()
)

volume_check

,volume_bucket,n,avg_trend
0,"(0.999, 39.0]",6041,-12.453986
1,"(39.0, 364.0]",5964,6.392526
2,"(364.0, 1375.0]",5997,1.288475
3,"(1375.0, 5167.6]",5998,-11.006163
4,"(5167.6, 517715.0]",6000,-11.395553


Verdict: MIXED

High volume pages have more opportunity but volume alone does not
guarantee decline. It should be combined with freshness.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [15]:
baseline = df.copy()


baseline["score"] = 0


# Staleness score
baseline.loc[
    baseline["days_since_last_update"] > 365,
    "score"
] += 40


baseline.loc[
    (baseline["days_since_last_update"] > 180) &
    (baseline["days_since_last_update"] <= 365),
    "score"
] += 25


# Traffic opportunity
baseline.loc[
    baseline["impressions_90d"] > baseline["impressions_90d"].median(),
    "score"
] += 20


# CTR weakness
baseline.loc[
    baseline["ctr"] < baseline["ctr"].median(),
    "score"
] += 20
def reason(row):

    reasons=[]

    if row["days_since_last_update"] > 180:
        reasons.append("STALE_CONTENT")

    if row["impressions_90d"] > df["impressions_90d"].median():
        reasons.append("HIGH_TRAFFIC_OPPORTUNITY")

    if row["ctr"] < df["ctr"].median():
        reasons.append("LOW_CTR")

    if len(reasons)==0:
        return "LOW_PRIORITY"

    return "_".join(reasons)


baseline["reason_code"] = baseline.apply(reason, axis=1)
def action(score):

    if score >= 60:
        return "refresh"

    elif score >= 30:
        return "monitor"

    else:
        return "leave"


baseline["action"] = baseline["score"].apply(action)
queue = baseline.sort_values(
    by="score",
    ascending=False
)



queue.head(10)



,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,staleness_bucket,volume_bucket,score,reason_code,action
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,informational,4329.0,27844.0,NaN,gemini-2.5-flash,4590,0,4,4,4,0,0,1,87,4,554,0,0,1831,0,3,231,181-365,5,194,181+,3500+,25000+,0.00,31.0,0.0,25.0,0.0,good,page_3_5,down,-69.7,181-365 days,"(1375.0, 5167.6]",65,STALE_CONTENT_HIGH_TRAFFIC_OPPORTUNITY_LOW_CTR,refresh
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.0,LOW,0.0,keyword article,transactional,3590.0,22780.0,NaN,gemini-2.5-flash,7812,1,5,5,5,0,0,2,69,5,292,1,1,2670,0,1,231,181-365,5,194,181+,3500+,15000-25000,0.01,39.0,0.0,40.0,0.0,good,page_3_5,down,-89.1,181-365 days,"(5167.6, 517715.0]",65,STALE_CONTENT_HIGH_TRAFFIC_OPPORTUNITY_LOW_CTR,refresh
18440,content_8d56efff1e71,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,1,0,2,2,2,0,0,1,1,2,1,0,0,0,0,1,372,365+,6,372,181+,NaN,NaN,0.00,35.0,0.0,50.0,0.0,low,page_3_5,new,NaN,365+ days,"(0.999, 39.0]",60,STALE_CONTENT_LOW_CTR,refresh
26242,content_55a5b1c46474,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,35,0,1,1,1,0,0,0,21,1,3,0,0,26,0,1,374,365+,6,373,181+,NaN,NaN,0.00,7.5,0.0,0.0,0.0,low,page_1,down,-88.5,365+ days,"(0.999, 39.0]",60,STALE_CONTENT_LOW_CTR,refresh
29384,content_f6fdf87348f6,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,2,0,1,1,1,0,0,0,1,1,0,0,1,2,0,0,373,365+,6,373,181+,NaN,NaN,0.00,32.5,0.0,0.0,0.0,low,page_3_5,down,-100.0,365+ days,"(0.999, 39.0]",60,STALE_CONTENT_LOW_CTR,refresh
24216,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.0,LOW,0.0,keyword article,informational,NaN,NaN,NaN,gpt-5-mini,2,0,2,2,2,0,0,1,1,2,0,0,0,2,0,1,372,365+,6,372,181+,NaN,NaN,0.00,7.0,0.0,50.0,0.0,low,page_1,down,-100.0,365+ days,"(0.999, 39.0]",60,STALE_CONTENT_LOW_CTR,refresh
6103,content_691fc6b910e6,client_6208ef0f77,0.0,0.0,LOW,0.0,keyword article,informational,6426.0,42872.0,NaN,gemini-3-flash-preview,56,0,18,9,9,0,0,0,30,5,22,0,7,25,0,2,236,181-365,5,236,181+,3500+,25000+,0.00,17.7,0.0,0.0,0.0,low,striking,stable,-12.0,181-365 days,"(39.0, 364.0]",45,STALE_CONTENT_LOW_CTR,monitor
2860,content_36e7b91747fa,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,972.0,7106.0,NaN,gpt-4o-mini,1,0,1,1,1,0,0,0,1,1,1,0,0,0,0,0,305,181-365,5,211,181+,<1000,<8000,0.00,0.0,0.0,0.0,0.0,low,top_3,new,NaN,181-365 days,"(0.999, 39.0]",45,STALE_CONTENT_LOW_CTR,monitor
2519,content_0edf498ae135,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,1034.0,7308.0,NaN,gpt-4o-mini,3,0,1,1,1,0,0,0,3,1,0,0,0,0,0,0,302,181-365,5,211,181+,1000-2000,<8000,0.00,2.3,0.0,0.0,0.0,low,top_3,flat,NaN,181-365 days,"(0.999, 39.0]",45,STALE_CONTENT_LOW_CTR,monitor
10836,content_e748f498b262,client_d4735e3a26,NaN,NaN,NaN,NaN,keyword article,NaN,1367.0,13108.0,NaN,gpt-4o-mini,2,0,2,2,2,0,0,2,2,2,1,0,0,0,0,0,304,181-365,5,211,181+,1000-2000,8000-15000,0.00,3.0,0.0,100.0,0.0,low,top_3,new,NaN,181-365 days,"(0.999, 39.0]",45,STALE_CONTENT_LOW_CTR,monitor


In [18]:
os.makedirs(
    "work/outputs",
    exist_ok=True
)
output = queue[
    [
        "content_id",
        "client_id",
        "score",
        "reason_code",
        "action"
    ]
]


output.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)


print("CSV created")

CSV created


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
top20 = queue.head(20)[
    [
        "content_id",
        "score",
        "reason_code",
        "action"
    ]
]


top20

,content_id,score,reason_code,action
698,content_b16bd7307b39,65,STALE_CONTENT_HIGH_TRAFFIC_OPPORTUNITY_LOW_CTR,refresh
11489,content_5feee3994adb,65,STALE_CONTENT_HIGH_TRAFFIC_OPPORTUNITY_LOW_CTR,refresh
18440,content_8d56efff1e71,60,STALE_CONTENT_LOW_CTR,refresh
26242,content_55a5b1c46474,60,STALE_CONTENT_LOW_CTR,refresh
29384,content_f6fdf87348f6,60,STALE_CONTENT_LOW_CTR,refresh
24216,content_1b4ec72dafd4,60,STALE_CONTENT_LOW_CTR,refresh
6103,content_691fc6b910e6,45,STALE_CONTENT_LOW_CTR,monitor
2860,content_36e7b91747fa,45,STALE_CONTENT_LOW_CTR,monitor
2519,content_0edf498ae135,45,STALE_CONTENT_LOW_CTR,monitor
10836,content_e748f498b262,45,STALE_CONTENT_LOW_CTR,monitor


## 4. Weak picks + leakage check



Weak picks:

Some pages may receive high scores because they are old and have traffic,
but age alone does not prove that updating will improve performance.

Leakage check:

The baseline does not use:
- trend_direction
- trend_pct
- is_declining_label

The score only uses information available before the decision point.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.